<a href="https://colab.research.google.com/github/Brunolms11/Calc_Numerico/blob/main/Atividade_Pr%C3%A1tica_III_Calculo_Numerico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Prática III — Cálculo Numérico

## Objetivo
Encontrar raízes aproximadas das quatro funções propostas usando os métodos de bisseção, Newton e secantes.

A implementação dos métodos foi mantida a partir dos códigos fornecidos:
- `bissecao`
- `newton`
- `metodo_secantes`

**Tolerância usada na atividade:** $10^{-6}$  
**Número máximo de iterações:** 100.


## 1. Funções da atividade

\[
f_1(x)=x^2-2
\]

\[
f_2(x)=x^3-x-2
\]

\[
f_3(x)=e^{-x}-x
\]

\[
f_4(x)=x^3-2x+2
\]

Para Newton:

\[
f'_1(x)=2x,\quad
f'_2(x)=3x^2-1,\quad
f'_3(x)=-e^{-x}-1,\quad
f'_4(x)=3x^2-2.
\]


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# ============================================================
# MÉTODO DE NEWTON — código fornecido
# ============================================================

def newton(f, df, x0, tol=1e-10, max_iter=50, deriv_tol=1e-14):
    """Resolve f(x)=0 pelo método de Newton e retorna raiz e histórico."""
    x = float(x0)
    historico = []

    for k in range(max_iter):
        fx = f(x)
        dfx = df(x)
        historico.append({"k": k, "x": x, "f(x)": fx, "f'(x)": dfx})

        if abs(fx) <= tol:
            return x, historico, "convergiu pelo resíduo"

        if abs(dfx) <= deriv_tol:
            return x, historico, "parou: derivada nula ou muito pequena"

        x_novo = x - fx / dfx

        # Um passo pequeno sem resíduo pequeno pode indicar estagnação.
        if abs(x_novo - x) <= tol * max(1.0, abs(x_novo)):
            x = x_novo
            if abs(f(x)) <= tol:
                historico.append({
                    "k": k + 1,
                    "x": x,
                    "f(x)": f(x),
                    "f'(x)": df(x)
                })
                return x, historico, "convergiu pelo resíduo após passo pequeno"
            return x, historico, "parou: passo pequeno, mas resíduo ainda alto"

        x = x_novo

    return x, historico, "parou: atingiu o máximo de iterações"


# ============================================================
# MÉTODO DAS SECANTES — código fornecido
# ============================================================

def metodo_secantes(f, x0, x1, tolerancia=1e-10, max_iter=50):
    """
    Aplica o método das secantes.

    Retorna:
    - as aproximações x_k;
    - os resíduos |f(x_k)|.
    """
    x_anterior = float(x0)
    x_atual = float(x1)

    aproximacoes = [x_anterior, x_atual]
    residuos = [abs(f(x_anterior)), abs(f(x_atual))]

    for k in range(max_iter):
        f_anterior = f(x_anterior)
        f_atual = f(x_atual)

        # Se já atingimos a tolerância, terminamos.
        if abs(f_atual) <= tolerancia:
            break

        denominador = f_atual - f_anterior

        # Evita divisão por um denominador muito pequeno.
        if abs(denominador) < 1e-14:
            print("Secantes parou: denominador nulo ou muito pequeno.")
            break

        # Calcula a próxima aproximação.
        x_novo = x_atual - f_atual * (x_atual - x_anterior) / denominador

        aproximacoes.append(x_novo)
        residuos.append(abs(f(x_novo)))

        # Desloca os pontos para a próxima iteração.
        x_anterior = x_atual
        x_atual = x_novo

        if abs(f(x_atual)) <= tolerancia:
            break

    return aproximacoes, residuos


# ============================================================
# MÉTODO DA BISSEÇÃO — código fornecido
# ============================================================

def bissecao(funcao, a, b, tolerancia=1e-8, max_iter=100):
    # Método da bisseção com registro das iterações.
    if a >= b:
        raise ValueError("É necessário fornecer a < b.")
    if tolerancia <= 0 or max_iter < 1:
        raise ValueError("Use tolerância positiva e max_iter >= 1.")

    fa, fb = funcao(a), funcao(b)

    if not np.isfinite([fa, fb]).all():
        raise ValueError("A função não é finita nas extremidades.")

    if fa == 0:
        return a, pd.DataFrame(), "a é raiz"

    if fb == 0:
        return b, pd.DataFrame(), "b é raiz"

    if fa * fb > 0:
        raise ValueError("Não há mudança de sinal nas extremidades.")

    historico = []

    for k in range(1, max_iter + 1):
        m = a + (b-a)/2
        fm = funcao(m)
        erro = (b-a)/2

        historico.append({
            "iteração": k,
            "a": a,
            "b": b,
            "m": m,
            "f(a)": fa,
            "f(m)": fm,
            "f(b)": fb,
            "erro máximo": erro
        })

        if fm == 0 or erro <= tolerancia:
            motivo = "raiz exata" if fm == 0 else "tolerância"
            break

        if fa * fm < 0:
            b, fb = m, fm
        else:
            a, fa = m, fm

    else:
        raise RuntimeError("Número máximo de iterações atingido.")

    return m, pd.DataFrame(historico), motivo


# ============================================================
# 1. FUNÇÕES DA ATIVIDADE
# ============================================================

def f1(x):
    return x**2 - 2

def df1(x):
    return 2*x


def f2(x):
    return x**3 - x - 2

def df2(x):
    return 3*x**2 - 1


def f3(x):
    return math.exp(-x) - x

def df3(x):
    return -math.exp(-x) - 1


def f4(x):
    return x**3 - 2*x + 2

def df4(x):
    return 3*x**2 - 2


# ============================================================
# 2. PARÂMETROS DA ATIVIDADE
# ============================================================

TOL = 1e-6
MAX_ITER = 100

funcoes = [
    ("f1", f1, df1, 1.5, 1, 2),
    ("f2", f2, df2, 1.5, 1, 2),
    ("f3", f3, df3, 0.5, 0, 1),
    ("f4", f4, df4, -2, -2, -1),
]


# ============================================================
# 3. EXECUÇÃO DOS MÉTODOS
# ============================================================

resultados = []
historicos = {}

for nome, f, df, x0, a, b in funcoes:

    # Bisseção
    raiz, historico, estado = bissecao(
        f, a, b, tolerancia=TOL, max_iter=MAX_ITER
    )

    resultados.append({
        "Função": nome,
        "Método": "Bisseção",
        "Valores iniciais": f"[{a}, {b}]",
        "Raiz aproximada": raiz,
        "Resíduo": abs(f(raiz)),
        "Iterações": len(historico),
        "Estado": estado
    })

    historicos[(nome, "Bisseção")] = historico

    # Newton
    raiz, historico, estado = newton(
        f, df, x0=x0, tol=TOL, max_iter=MAX_ITER
    )

    resultados.append({
        "Função": nome,
        "Método": "Newton",
        "Valores iniciais": f"x0={x0}",
        "Raiz aproximada": raiz,
        "Resíduo": abs(f(raiz)),
        "Iterações": len(historico),
        "Estado": estado
    })

    historicos[(nome, "Newton")] = historico

    # Secantes
    aproximacoes, residuos = metodo_secantes(
        f, a, b, tolerancia=TOL, max_iter=MAX_ITER
    )

    raiz = aproximacoes[-1]

    resultados.append({
        "Função": nome,
        "Método": "Secantes",
        "Valores iniciais": f"({a}, {b})",
        "Raiz aproximada": raiz,
        "Resíduo": abs(f(raiz)),
        "Iterações": len(aproximacoes) - 2,
        "Estado": (
            "convergiu pelo resíduo"
            if abs(f(raiz)) <= TOL
            else "parou sem atingir a tolerância"
        )
    })

    historicos[(nome, "Secantes")] = (aproximacoes, residuos)


# ============================================================
# 4. TABELA FINAL
# ============================================================

tabela_final = pd.DataFrame(resultados)

display(
    tabela_final.style.format({
        "Raiz aproximada": "{:.12f}",
        "Resíduo": "{:.6e}"
    })
)


# ============================================================
# 5. HISTÓRICOS INDIVIDUAIS
# ============================================================

# Exemplo: histórico de Newton para f1
raiz, hist, estado = newton(
    f1, df1, x0=1.5, tol=TOL, max_iter=MAX_ITER
)

print("f1 — Newton")
print("Raiz aproximada:", raiz)
print("Resíduo:", abs(f1(raiz)))
print("Iterações registradas:", len(hist))
print("Estado:", estado)

display(pd.DataFrame(hist))


# ============================================================
# 6. GRÁFICOS DAS FUNÇÕES
# ============================================================

limites = {
    "f1": (0.5, 2.2),
    "f2": (0.8, 2.1),
    "f3": (-0.2, 1.2),
    "f4": (-2.2, 0.5),
}

for nome, f, df, x0, a, b in funcoes:
    lo, hi = limites[nome]
    xs = np.linspace(lo, hi, 600)

    plt.figure(figsize=(8, 5))
    plt.plot(xs, [f(x) for x in xs], linewidth=2, label=f"{nome}(x)")
    plt.axhline(0, linewidth=1, label="y=0")
    plt.scatter([a, b], [f(a), f(b)])

    plt.xlabel("x")
    plt.ylabel("f(x)")
    plt.title(f"Gráfico de {nome}")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.show()


# ============================================================
# 7. GIF — BISSEÇÃO
# ============================================================

def gif_bissecao(f, nome, a, b, arquivo):
    raiz, hist, estado = bissecao(
        f, a, b, tolerancia=TOL, max_iter=MAX_ITER
    )

    lo, hi = limites[nome]
    xs = np.linspace(lo, hi, 600)

    fig, ax = plt.subplots(figsize=(8, 5))

    def atualizar(i):
        ax.clear()

        linha = hist.iloc[i]
        a_i = linha["a"]
        b_i = linha["b"]
        m = linha["m"]

        ax.plot(xs, [f(x) for x in xs], linewidth=2, label="f(x)")
        ax.axhline(0, linewidth=1, label="y=0")

        ax.plot(
            [a_i, b_i],
            [0, 0],
            linewidth=5,
            alpha=0.5,
            label="intervalo"
        )

        ax.scatter(
            [a_i, m, b_i],
            [f(a_i), f(m), f(b_i)],
            s=60
        )

        ax.axvline(m, linestyle="--", alpha=0.7)

        ax.set_xlim(lo, hi)
        ax.set_xlabel("x")
        ax.set_ylabel("f(x)")
        ax.set_title(
            f"Bisseção — {nome} — iteração {i+1}\n"
            f"m={m:.8f} | |f(m)|={abs(f(m)):.3e}"
        )
        ax.grid(True, alpha=0.25)
        ax.legend()

    anim = FuncAnimation(
        fig,
        atualizar,
        frames=len(hist),
        interval=650,
        repeat=True
    )

    anim.save(arquivo, writer=PillowWriter(fps=1.5))
    plt.close(fig)


# Exemplo:
# gif_bissecao(f2, "f2", 1, 2, "GIF_bissecao_f2.gif")


# ============================================================
# 8. GIF — NEWTON
# ============================================================

def gif_newton(f, df, nome, x0, arquivo):
    raiz, hist, estado = newton(
        f, df, x0=x0, tol=TOL, max_iter=MAX_ITER
    )

    lo, hi = limites[nome]
    xs = np.linspace(lo, hi, 600)

    fig, ax = plt.subplots(figsize=(8, 5))

    def atualizar(i):
        ax.clear()

        linha = hist[i]
        x_i = linha["x"]
        y_i = f(x_i)
        inclinacao = df(x_i)

        ax.plot(xs, [f(x) for x in xs], linewidth=2, label="f(x)")
        ax.axhline(0, linewidth=1, label="y=0")

        tangente = y_i + inclinacao * (xs - x_i)

        ax.plot(
            xs,
            tangente,
            linestyle="--",
            label=f"tangente k={linha['k']}"
        )

        ax.scatter([x_i], [y_i], s=70)

        if abs(inclinacao) > 1e-14:
            x_novo = x_i - y_i / inclinacao
            ax.scatter([x_novo], [0], s=90)
            ax.plot(
                [x_i, x_novo],
                [y_i, 0],
                linestyle=":",
                alpha=0.7
            )

        ax.set_xlim(lo, hi)
        ax.set_xlabel("x")
        ax.set_ylabel("f(x)")
        ax.set_title(
            f"Newton — {nome} — k={linha['k']}\n"
            f"x={x_i:.8f} | |f(x)|={abs(y_i):.3e}"
        )
        ax.grid(True, alpha=0.25)
        ax.legend()

    anim = FuncAnimation(
        fig,
        atualizar,
        frames=len(hist),
        interval=700,
        repeat=True
    )

    anim.save(arquivo, writer=PillowWriter(fps=1.4))
    plt.close(fig)


# Exemplo:
# gif_newton(f3, df3, "f3", 0.5, "GIF_newton_f3.gif")


# ============================================================
# 9. GIF — SECANTES
# ============================================================

def gif_secantes(f, nome, x0, x1, arquivo):
    aproximacoes, residuos = metodo_secantes(
        f, x0, x1, tolerancia=TOL, max_iter=MAX_ITER
    )

    lo, hi = limites[nome]
    xs = np.linspace(lo, hi, 600)

    fig, ax = plt.subplots(figsize=(8, 5))

    def atualizar(i):
        ax.clear()

        a = aproximacoes[i]
        b = aproximacoes[i+1]

        fa = f(a)
        fb = f(b)

        ax.plot(xs, [f(x) for x in xs], linewidth=2, label="f(x)")
        ax.axhline(0, linewidth=1, label="y=0")

        ax.scatter(
            [a, b],
            [fa, fb],
            s=65
        )

        if abs(b-a) > 0:
            inclinacao = (fb-fa)/(b-a)
            reta = fb + inclinacao * (xs-b)

            ax.plot(
                xs,
                reta,
                linestyle="--",
                label="reta secante"
            )

            x_novo = b - fb/inclinacao

            ax.scatter([x_novo], [0], s=90)

            ax.plot(
                [b, x_novo],
                [fb, 0],
                linestyle=":",
                alpha=0.7
            )

        ax.set_xlim(lo, hi)
        ax.set_xlabel("x")
        ax.set_ylabel("f(x)")
        ax.set_title(
            f"Secantes — {nome} — passo {i+1}\n"
            f"|f(x atual)|={abs(fb):.3e}"
        )
        ax.grid(True, alpha=0.25)
        ax.legend()

    anim = FuncAnimation(
        fig,
        atualizar,
        frames=len(aproximacoes)-1,
        interval=700,
        repeat=True
    )

    anim.save(arquivo, writer=PillowWriter(fps=1.4))
    plt.close(fig)


# Exemplos para cumprir a atividade:
# gif_bissecao(f2, "f2", 1, 2, "GIF_1_bissecao_f2.gif")
# gif_newton(f3, df3, "f3", 0.5, "GIF_2_newton_f3.gif")
# gif_secantes(f4, "f4", -2, -1, "GIF_3_secantes_f4.gif")


## 2. Análise dos resultados

A bisseção preserva um intervalo com mudança de sinal e, no código fornecido, para quando a meia-largura do intervalo satisfaz a tolerância. Por isso, o resíduo final pode ser maior que $10^{-6}$ mesmo quando o método é considerado convergente pelo critério adotado.

Newton usa a derivada e interrompe quando o resíduo fica abaixo da tolerância, além de proteger a divisão por derivada muito pequena.

As secantes não usam a derivada explicitamente. O código mantém duas aproximações e interrompe quando o resíduo satisfaz a tolerância ou quando o denominador fica muito pequeno.

Os resultados devem ser analisados em conjunto: raiz aproximada, resíduo, número de iterações e critério de parada.


## 3. Tabela final\n\n| Função   | Método   | Valores iniciais   |   Raiz aproximada |     Resíduo |   Iterações | Resultado                                 |
|:---------|:---------|:-------------------|------------------:|------------:|------------:|:------------------------------------------|
| f1       | Bisseção | [1, 2]             |          1.41421  | 1.61742e-06 |          20 | tolerância                                |
| f1       | Newton   | x0=1.5             |          1.41421  | 4.51061e-12 |           4 | convergiu pelo resíduo                    |
| f1       | Secantes | (1, 2)             |          1.41421  | 8.93146e-10 |           5 | convergência pelo resíduo                 |
| f2       | Bisseção | [1, 2]             |          1.52138  | 4.26583e-06 |          20 | tolerância                                |
| f2       | Newton   | x0=1.5             |          1.52138  | 5.89387e-07 |           3 | convergiu pelo resíduo                    |
| f2       | Secantes | (1, 2)             |          1.52138  | 7.01548e-09 |           6 | convergência pelo resíduo                 |
| f3       | Bisseção | [0, 1]             |          0.567143 | 2.34816e-07 |          20 | tolerância                                |
| f3       | Newton   | x0=0.5             |          0.567143 | 1.9648e-07  |           3 | convergiu pelo resíduo                    |
| f3       | Secantes | (0, 1)             |          0.567143 | 2.53802e-08 |           4 | convergência pelo resíduo                 |
| f4       | Bisseção | [-2, -1]           |         -1.76929  | 3.52184e-06 |          20 | tolerância                                |
| f4       | Newton   | x0=-2              |         -1.76929  | 5.05374e-13 |           5 | convergiu pelo resíduo após passo pequeno |
| f4       | Secantes | (-2, -1)           |         -1.76929  | 5.5937e-07  |           6 | convergência pelo resíduo                 |

## 4. Animações

Foram produzidas três animações:
1. Bisseção aplicada a $f_2$;
2. Newton aplicado a $f_3$;
3. Secantes aplicada a $f_4$.

Cada animação mostra a função, o eixo $y=0$ e a construção geométrica do método.
